# Radical Optimization - Simplified Approach for 1B Model

## Your Results So Far:
- k=20: 29.66% F1
- k=25: 31.46% F1  
- k=30: 32.12% F1 (best)

## Problem Identified:
LLM extraction is the bottleneck. Even with answers in documents, the 1B model struggles to extract them from long contexts.

## New Strategy:
1. Much smaller contexts (3-5 docs instead of 30)
2. Ultra-simple prompts
3. Very low temperature (0.05 or 0.01)
4. Force short answers (max 20 tokens)

Target: 35-40% F1

In [ ]:
# Setup
import pandas as pd
import json
import re
import string
from collections import Counter

from pyserini.search import SimpleSearcher
import transformers
import torch

searcher = SimpleSearcher.from_prebuilt_index('wikipedia-kilt-doc')
print("Index loaded")

In [ ]:
# Load data
from google.colab import drive
drive.mount('/content/drive')

train_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/train.csv"
test_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/test.csv"

df_train = pd.read_csv(train_path, converters={"answers": json.loads})
df_test = pd.read_csv(test_path)

print(f"Train: {len(df_train)}, Test: {len(df_test)}")

In [ ]:
# Load LLM
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")

model_id = "meta-llama/Llama-3.2-1B-Instruct"
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("LLM loaded")

## Query Expansion & Retrieval (Keep What Works!)

In [ ]:
def expand_query(query):
    """
    Generate query variations - this helped get to 32%!
    """
    queries = []
    
    # 1. Original query (just remove ?)
    queries.append(query.rstrip('?').strip())
    
    # 2. Lowercase version
    lower_q = query.lower().rstrip('?').strip()
    if lower_q != queries[0]:
        queries.append(lower_q)
    
    # 3. Extract subject after question words
    pattern = r'^(what|where|when|who|which|how)\s+(is|are|was|were|does|do|did)\s+(the\s+)?(.+)$'
    match = re.match(pattern, lower_q, re.IGNORECASE)
    if match:
        subject = match.group(4).strip()
        if subject and subject not in queries:
            queries.append(subject)
    
    # 4. Keywords only (remove stop words)
    minimal_stops = {'what', 'where', 'when', 'who', 'which', 'how', 'is', 'are', 'was', 'were', 'do', 'does', 'did'}
    tokens = lower_q.split()
    important_tokens = [t for t in tokens if t not in minimal_stops and len(t) > 2]
    if len(important_tokens) >= 2:
        important_q = ' '.join(important_tokens)
        if important_q not in queries:
            queries.append(important_q)
    
    return queries


def retrieve_documents(question, k=30, mu=1000):
    """
    Retrieve with query expansion and fusion - THIS WORKS (got you to 32%)!
    """
    searcher.set_qld(mu=mu)
    
    # Generate query variations
    query_variations = expand_query(question)
    
    doc_info = {}
    
    # Search with each variation
    for q_var in query_variations:
        hits = searcher.search(q_var, k)
        
        for hit in hits:
            doc_id = hit.docid
            
            if doc_id not in doc_info:
                doc = searcher.doc(doc_id)
                data = json.loads(doc.raw())
                
                doc_info[doc_id] = {
                    'content': data.get('contents', ''),
                    'title': data.get('title', ''),
                    'score': hit.score,
                    'count': 1
                }
            else:
                # Boost documents appearing in multiple queries
                doc_info[doc_id]['score'] += hit.score * 0.8
                doc_info[doc_id]['count'] += 1
    
    # Sort by fused score
    sorted_docs = sorted(doc_info.values(), key=lambda x: x['score'], reverse=True)
    
    return sorted_docs

## Prompt Strategy 1: Minimal (Top 3 docs, ultra-simple)

In [ ]:
def create_minimal_prompt(question, docs):
    """Smallest context, simplest prompt."""
    # Use only top 3 docs, 300 chars each
    context_parts = []
    for doc in docs[:3]:
        text = doc['content'].replace('\n', ' ')[:300]
        context_parts.append(f"{doc['title']}: {text}")
    
    context = '\n\n'.join(context_parts)
    
    # Just user message
    user = (
        f"{context}\n\n"
        f"Answer this question with 1-3 words:\n"
        f"{question}\n\n"
        f"Answer:"
    )
    
    return [{"role": "user", "content": user}]

## Prompt Strategy 2: QA Format (Top 5 docs)

In [ ]:
def create_qa_prompt(question, docs):
    """Classic QA format - familiar to LLM."""
    # Top 5 docs, 350 chars each
    context_parts = []
    for doc in docs[:5]:
        text = doc['content'].replace('\n', ' ')[:350]
        context_parts.append(f"{doc['title']}: {text}")
    
    context = '\n'.join(context_parts)
    
    messages = [
        {"role": "system", "content": "Answer with 1-5 words only."},
        {"role": "user", "content": f"{context}\n\nQ: {question}\nA:"}
    ]
    
    return messages

## Prompt Strategy 3: Direct Extraction (Top 7 docs)

In [ ]:
def create_direct_prompt(question, docs):
    """Extraction-focused."""
    # Top 7 docs, 400 chars each
    context = '\n'.join([
        f"{doc['title']}: {doc['content'][:400]}"
        for doc in docs[:7]
    ])
    
    messages = [
        {"role": "system", "content": "Extract the answer from the text. 1-5 words only."},
        {"role": "user", "content": f"Text: {context}\n\nQuestion: {question}\n\nShort answer:"}
    ]
    
    return messages

## Answer Extraction

In [ ]:
def clean_answer(text):
    """Extract just the answer."""
    if not text:
        return 'unknown'
    
    text = text.strip()
    
    # Remove prefixes
    prefixes = ['the answer is', 'answer:', 'a:', 'short answer:', 'based on', 'according to']
    text_lower = text.lower()
    for prefix in prefixes:
        if text_lower.startswith(prefix):
            text = text[len(prefix):].strip()
            text_lower = text.lower()
    
    # Remove quotes
    text = text.strip('"').strip()
    
    # Take first sentence
    if '.' in text:
        sentences = text.split('.')
        for s in sentences:
            s = s.strip()
            if s and len(s.split()) >= 1:
                text = s
                break
    
    # If too long, truncate
    words = text.split()
    if len(words) > 10:
        text = ' '.join(words[:5])
    
    # Check for unknown
    if text.lower() in ['unknown', 'i dont know', "i don't know", 'not found']:
        return 'unknown'
    
    return text.strip()

## Main Pipeline

In [ ]:
def answer_question(question, k=30, mu=1000, temperature=0.05, prompt_style='minimal'):
    """Generate answer with specified strategy."""
    docs = retrieve_documents(question, k=k, mu=mu)
    
    if not docs:
        return 'unknown'
    
    # Select prompt style
    if prompt_style == 'minimal':
        messages = create_minimal_prompt(question, docs)
    elif prompt_style == 'qa':
        messages = create_qa_prompt(question, docs)
    else:
        messages = create_direct_prompt(question, docs)
    
    # Generate with strict limits
    try:
        outputs = pipeline(
            messages,
            max_new_tokens=30,
            eos_token_id=terminators,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
        )
        
        raw = outputs[0]["generated_text"][-1].get('content', 'unknown')
        return clean_answer(raw)
    
    except Exception as e:
        return 'unknown'

## Evaluation

In [ ]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return int(pred_tokens == gt_tokens)
    if num_same == 0:
        return 0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def evaluate(predictions_dict, df_gold):
    f1_sum = 0.0
    count = 0
    
    for idx, row in df_gold.iterrows():
        qid = row['id']
        if qid not in predictions_dict:
            continue
        
        prediction = predictions_dict[qid]
        ground_truths = row['answers']
        
        f1 = max(f1_score(prediction, gt) for gt in ground_truths)
        f1_sum += f1
        count += 1
    
    return (100.0 * f1_sum / count) if count > 0 else 0.0

## Test Different Strategies

In [ ]:
# Test on sample
sample_size = 100
df_sample = df_train.sample(n=sample_size, random_state=42)

print(f"Testing strategies on {sample_size} examples...\n")

configs = [
    # Minimal context strategies
    {'k': 30, 'mu': 1000, 'temp': 0.05, 'style': 'minimal'},
    {'k': 30, 'mu': 1000, 'temp': 0.01, 'style': 'minimal'},
    
    # QA format
    {'k': 30, 'mu': 1000, 'temp': 0.05, 'style': 'qa'},
    
    # Direct extraction
    {'k': 30, 'mu': 1000, 'temp': 0.05, 'style': 'direct'},
    
    # Try fewer retrieved docs
    {'k': 20, 'mu': 1000, 'temp': 0.05, 'style': 'minimal'},
    
    # Try more retrieved docs
    {'k': 40, 'mu': 1000, 'temp': 0.05, 'style': 'minimal'},
]

best_score = 0
best_config = None

for i, config in enumerate(configs):
    print(f"\nConfig {i+1}/{len(configs)}: {config}")
    
    predictions = {}
    for idx, row in df_sample.iterrows():
        predictions[row['id']] = answer_question(
            row['question'],
            k=config['k'],
            mu=config['mu'],
            temperature=config['temp'],
            prompt_style=config['style']
        )
    
    score = evaluate(predictions, df_sample)
    print(f"F1 Score: {score:.2f}%")
    
    if score > best_score:
        best_score = score
        best_config = config
        print("  ⭐ New best!")

print("\n" + "=" * 80)
print(f"Best: {best_config}")
print(f"Score: {best_score:.2f}%")
print(f"Improvement from 32.12%: {best_score - 32.12:+.2f}%")
print("=" * 80)

## Generate Test Predictions

In [ ]:
# Use best config from above
FINAL_K = 30
FINAL_MU = 1000
FINAL_TEMP = 0.05
FINAL_STYLE = 'minimal'

print(f"Generating predictions with: k={FINAL_K}, mu={FINAL_MU}, temp={FINAL_TEMP}, style={FINAL_STYLE}\n")

test_predictions = {}

for idx, row in df_test.iterrows():
    answer = answer_question(
        row['question'],
        k=FINAL_K,
        mu=FINAL_MU,
        temperature=FINAL_TEMP,
        prompt_style=FINAL_STYLE
    )
    test_predictions[row['id']] = answer
    
    if (idx + 1) % 100 == 0:
        print(f"  {idx + 1}/{len(df_test)}")

print("\nDone!")

## Save Predictions

In [ ]:
# Format and save
submission_df = pd.DataFrame([
    {'id': qid, 'prediction': answer}
    for qid, answer in test_predictions.items()
])

submission_df = submission_df.sort_values('id').reset_index(drop=True)

# CRITICAL: Format with ensure_ascii=False
submission_df['prediction'] = submission_df['prediction'].apply(
    lambda x: json.dumps([x], ensure_ascii=False)
)

output_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/radical_predictions.csv"
submission_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total: {len(submission_df)}")
print("\nFirst 3:")
print(submission_df.head(3))